In [ ]:
def interactive_chat():
    \"\"\"
    Interactive chat interface for the production travel assistant.
    \"\"\"
    thread_id = str(uuid.uuid4())[:8]
    print(f"🎉 Welcome to the Cornwall Travel Assistant!")
    print(f"🆔 Session ID: {thread_id}")
    print(f"💬 I can help with travel information, weather, and accommodation in Cornwall.")
    print(f"💡 Try asking about: destinations, weather, hotels, or BnB availability")
    print(f"❌ Type 'exit', 'quit', or 'stop' to end the conversation\\n")
    
    while True:
        try:
            user_input = input("🗣️ You: ").strip()
            
            if user_input.lower() in {"exit", "quit", "stop", ""}:
                print("👋 Thank you for using the Cornwall Travel Assistant! Safe travels!")
                break
                
            # Process the query
            state = {"messages": [HumanMessage(content=user_input)]}
            config = {"configurable": {"thread_id": thread_id}}
            
            print("\\n🤔 Thinking...")
            result = production_travel_assistant.invoke(state, config=config)
            
            response_msg = result["messages"][-1]
            print(f"\\n🤖 Assistant: {response_msg.content}\\n")
            print("-" * 60)
            
        except KeyboardInterrupt:
            print("\\n👋 Chat interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")
            print("Please try again with a different question.\\n")

# Uncomment the line below to start interactive chat:
# interactive_chat()

print("🎮 Interactive chat ready! Uncomment the last line above to start chatting!")

## 🎉 Tutorial Complete! 

Congratulations! You've successfully built a **production-ready multi-agent travel assistant** with:

### ✅ **Core Features Implemented:**
1. **Vector-based Knowledge Retrieval** - WikiVoyage content search
2. **Multi-Agent Architecture** - Specialized agents for different domains
3. **Intelligent Routing** - LLM-powered query classification
4. **Guardrails** - Content filtering and refusal handling
5. **Database Integration** - Hotel booking with mock database
6. **Persistent Memory** - Conversation state across interactions
7. **Weather Forecasting** - Mock weather service integration
8. **BnB Booking** - Accommodation search and availability

### 🏗️ **Architecture Evolution:**
- **Part 1**: Simple single-agent (~260 lines)
- **Part 2**: Multi-agent with routing (~450 lines) 
- **Part 3**: Production system with guardrails (~500+ lines)

### 🎯 **Key Learning Outcomes:**
1. **LangGraph Mastery**: State management, nodes, edges, routing
2. **Agent Design Patterns**: Specialization, tool binding, prompting
3. **Production Considerations**: Guardrails, error handling, persistence
4. **RAG Implementation**: Vector search, embeddings, retrieval
5. **Database Integration**: SQL tools, structured queries
6. **System Scalability**: From prototype to production-ready

### 🚀 **Next Steps:**
- Deploy as a web application with FastAPI/Streamlit
- Add real weather API integration (AccuWeather, OpenWeather)
- Implement actual hotel booking workflows
- Add more specialized agents (restaurant recommendations, activity booking)
- Enhance with voice interface capabilities
- Add analytics and monitoring

### 🔧 **Production Deployment:**
To convert this notebook to a production Python script:
1. Extract the final system (Part 3) code
2. Add proper error handling and logging
3. Implement FastAPI endpoints
4. Add configuration management
5. Set up database persistence
6. Add monitoring and observability

---

### 🏆 **Interactive Chat Interface**

Want to try the full system? Run the cell below for an interactive chat session:"

In [ ]:
# Test 4: Follow-up question with memory (using same thread)
test_production_system("What about accommodation options in those coastal towns?", thread_id=thread1)

In [ ]:
# Test 3: Guardrail test - non-travel query (should be refused)
test_production_system("Can you help me write Python code for data analysis?")

In [ ]:
# Test 2: Accommodation query
test_production_system("I need a hotel in Newquay for this weekend. What are my options and prices?")

In [ ]:
# Test 1: Valid travel query
result1, thread1 = test_production_system("What's the weather like in Cornwall and what are the best coastal towns to visit?")

In [ ]:
def test_production_system(query: str, thread_id: str = None):
    \"\"\"
    Test the production system with persistent memory.
    \"\"\"
    if not thread_id:
        thread_id = str(uuid.uuid4())[:8]  # Short ID for readability
    
    print(f"\\n🗣️ User ({thread_id}): {query}")
    print("="*80)
    
    # Create state with conversation config
    state = {"messages": [HumanMessage(content=query)]}
    config = {"configurable": {"thread_id": thread_id}}
    
    # Invoke with persistent memory
    result = production_travel_assistant.invoke(state, config=config)
    
    # Get response
    response_msg = result["messages"][-1]
    print(f"🤖 Assistant: {response_msg.content}")
    print("="*80)
    
    return result, thread_id

## Step 23: Test Production System

Let's thoroughly test our production-ready system with various scenarios.

In [ ]:
# Build the advanced multi-agent graph
graph_v3 = StateGraph(AgentStateV2)

# Add all nodes
graph_v3.add_node("router_agent", advanced_router_agent_node)
graph_v3.add_node("travel_info_agent", travel_info_agent_v3)
graph_v3.add_node("accommodation_booking_agent", accommodation_booking_agent_v3)
graph_v3.add_node("guardrail_refusal", guardrail_refusal_node)

# Add edges to END
graph_v3.add_edge("travel_info_agent", END)
graph_v3.add_edge("accommodation_booking_agent", END)
graph_v3.add_edge("guardrail_refusal", END)

# Set entry point
graph_v3.set_entry_point("router_agent")

# Add persistent memory
checkpointer = InMemorySaver()
production_travel_assistant = graph_v3.compile(checkpointer=checkpointer)

print("🏗️ Production-ready travel assistant compiled!")
print("📊 Graph structure:")
print("   User → router_agent → [guardrail_check] → [specialist_agent | refusal] → END")
print("💾 Memory: Persistent conversation state enabled")

In [ ]:
# Enhanced tools list with hotel search
TOOLS_V3 = [search_travel_info, weather_forecast, check_bnb_availability, search_hotels]

# Enhanced travel info agent
travel_info_agent_v3 = create_react_agent(
    model=llm_model_v2,
    tools=[search_travel_info, weather_forecast],
    state_schema=AgentStateV2,
    prompt=(
        "You are a knowledgeable Cornwall travel expert. You can search for detailed travel information "
        "about destinations, attractions, activities, and provide weather forecasts for towns in Cornwall. "
        "Always provide specific, helpful recommendations based on the information you find. "
        "If asked about accommodations, suggest they ask about hotel or BnB availability specifically."
    )
)

# Enhanced accommodation booking agent  
accommodation_booking_agent_v3 = create_react_agent(
    model=llm_model_v2,
    tools=[check_bnb_availability, search_hotels, search_travel_info],
    state_schema=AgentStateV2,
    prompt=(
        "You are a Cornwall accommodation specialist. You can search for both BnB availability "
        "and hotel options with detailed pricing information. Always provide specific details "
        "about available rooms, pricing, and locations. You can also search for travel information "
        "to help guests choose the best location for their needs."
    )
)

print("✅ Enhanced specialized agents ready!")
print("   🗺️ Travel Agent: Enhanced with weather + comprehensive travel info")
print("   🏨 Accommodation Agent: BnB + Hotel search + location context")

## Step 22: Build Production-Ready System

Now we'll create the final, production-ready graph with all features.

In [ ]:
def advanced_router_agent_node(state: AgentStateV2) -> Command[str]:
    \"\"\"
    Advanced router node with guardrail checking.
    
    This function:
    1. Checks if the query is travel-related (guardrail)
    2. If not travel-related, returns a polite refusal
    3. If travel-related, routes to the appropriate specialist agent
    \"\"\"
    messages = state["messages"]
    last_msg = messages[-1] if messages else None
    
    if isinstance(last_msg, HumanMessage):
        user_input = last_msg.content
        print(f"🛡️ Checking guardrails for: '{user_input[:50]}{'...' if len(user_input) > 50 else ''}'")
        
        # Step 1: Guardrail classification
        guardrail_messages = [
            SystemMessage(content=GUARDRAIL_SYSTEM_PROMPT),
            HumanMessage(content=user_input),
        ]
        decision = llm_guardrail.invoke(guardrail_messages)
        print(f"🎯 Guardrail decision: {decision.is_travel} - {decision.explanation}")
        
        # Step 2: Handle non-travel queries
        if not decision.is_travel:
            refusal_text = (
                "I'm sorry, but I can only help with travel-related questions about Cornwall, England. "
                "I can assist with destinations, attractions, accommodations, weather, and activities in Cornwall. "
                "Please feel free to ask me about your Cornwall travel plans!"
            )
            print(f"❌ Query refused: not travel-related")
            return Command(
                update={"messages": [AIMessage(content=refusal_text)]},
                goto="guardrail_refusal",
            )
        
        # Step 3: Route travel-related queries
        print(f"✅ Query approved, routing...")
        router_messages = [
            SystemMessage(content=ROUTER_SYSTEM_PROMPT),
            HumanMessage(content=user_input)
        ]
        router_response = llm_router.invoke(router_messages)
        agent_name = router_response.agent.value
        
        print(f"🎯 Routing to: {agent_name}")
        return Command(update=state, goto=agent_name)
    
    # Default routing
    return Command(update=state, goto=AgentType.travel_info_agent.value)

def guardrail_refusal_node(state: AgentStateV2):
    \"\"\"Simple node that ends the conversation after a guardrail refusal.\"\"\"
    return {}

print("🚦 Advanced router with guardrails ready!")

## Step 21: Enhanced Router with Guardrails

Let's create an advanced router that includes guardrail checking and refusal handling.

In [ ]:
# Guardrail decision model
class GuardrailDecision(BaseModel):
    is_travel: bool = Field(..., description="Is this query related to travel, tourism, or Cornwall?")
    explanation: str = Field(..., description="Brief explanation of the decision")

# Guardrail system prompts
GUARDRAIL_SYSTEM_PROMPT = (
    "You are a content classifier. Determine if the user's message is related to travel, tourism, "
    "destinations, accommodations, weather, or activities in Cornwall/England. "
    "Return true if it's travel-related, false otherwise."
)

AGENT_REFUSAL_INSTRUCTION = (
    "The user's request is not travel-related or not about Cornwall/England. "
    "Politely refuse and briefly explain what topics you can help with "
    "(destinations, attractions, accommodations, weather in Cornwall)."
)

# Create guardrail LLM
llm_guardrail = llm_model_v2.with_structured_output(GuardrailDecision)

print("🛡️ Guardrail system configured!")

## Step 20: Create Guardrail System

Now we'll add guardrails to ensure conversations stay focused on travel-related topics.

In [ ]:
# Additional imports for advanced system
import uuid
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# Create a simple hotel search tool (mock database for demo)
@tool(description="Search for hotels in Cornwall towns with pricing and availability.")
def search_hotels(town: str = "", max_price: float = 1000.0) -> str:
    \"\"\"Search for hotels in Cornwall. Optionally filter by town and maximum price.\"\"\"
    
    # Mock hotel database
    hotels_db = {
        "Newquay": [("Seaview Hotel", 4.5, 120, 180), ("Atlantic Bay Hotel", 4.2, 95, 150)],
        "Falmouth": [("Harbour Inn", 4.2, 95, 150), ("Maritime Hotel", 4.0, 110, 170)], 
        "St Ives": [("St Ives Bay Resort", 4.8, 140, 210), ("Coastal View Hotel", 4.3, 100, 160)],
        "Penzance": [("Penzance Palace", 4.7, 130, 200), ("Sea Breeze Hotel", 4.1, 90, 140)],
        "St Austell": [("Cornish Retreat", 4.0, 110, 170), ("Garden Hotel", 4.2, 105, 165)]
    }
    
    results = []
    for location, hotels in hotels_db.items():
        if not town or town.lower() in location.lower():
            for name, rating, single, double in hotels:
                if single <= max_price or double <= max_price:
                    results.append(f"{name} in {location} (★{rating}) - Single: £{single}, Double: £{double}")
    
    if not results:
        return f"No hotels found in {town} under £{max_price}"
    
    return "\\n".join(results[:5])  # Limit to 5 results

print("🏨 Hotel database integration ready!")

## 🎉 Part 2 Complete!

Excellent progress! You've now built a sophisticated multi-agent system with:
- ✅ **Intelligent routing** based on query type
- ✅ **Multiple specialized agents** for different domains  
- ✅ **Weather forecasting** capabilities
- ✅ **Accommodation booking** functionality
- ✅ **Structured outputs** for reliable routing decisions

### What we learned:
1. **Agent Specialization**: How to create domain-specific agents
2. **Intelligent Routing**: Using LLMs to make routing decisions
3. **Structured Outputs**: Ensuring reliable, typed responses from LLMs
4. **Graph Complexity**: Managing more complex agent workflows

---

# Part 3: Advanced System with Guardrails & Database 🛡️

In this final part, we'll add production-ready features:
- **Guardrails** to keep conversations on-topic
- **SQLite database** integration for hotel bookings
- **Persistent memory** across conversations
- **Advanced routing** with refusal handling

## Step 19: Database Setup

Let's add real database integration for hotel bookings.

In [ ]:
# Test 3: General travel info (should route to travel_info_agent)
test_multi_agent("What are the main attractions in Penzance?")

In [ ]:
# Test 2: Accommodation query (should route to accommodation_booking_agent)
test_multi_agent("I need accommodation for 2 people in St Ives. What's available?")

In [ ]:
# Test 1: Travel information query (should route to travel_info_agent)
test_multi_agent("What's the weather like in Newquay today?")

In [ ]:
def test_multi_agent(query: str):
    \"\"\"
    Test the multi-agent system with routing.
    \"\"\"
    print(f"\\n🗣️ User: {query}")
    print("="*70)
    
    # Create initial state
    state = {"messages": [HumanMessage(content=query)]}
    
    # Invoke the multi-agent system
    result = multi_agent_system.invoke(state)
    
    # Get the final response
    response_msg = result["messages"][-1]
    print(f"🤖 Assistant: {response_msg.content}")
    print("="*70)
    
    return result

## Step 18: Test Multi-Agent System

Let's test our enhanced system with various types of queries to see how routing works.

In [ ]:
# Build the multi-agent graph
graph_v2 = StateGraph(AgentStateV2)

# Add all nodes
graph_v2.add_node("router_agent", router_agent_node)
graph_v2.add_node("travel_info_agent", travel_info_agent_v2)
graph_v2.add_node("accommodation_booking_agent", accommodation_booking_agent)

# Add edges from specialized agents to END
graph_v2.add_edge("travel_info_agent", END)
graph_v2.add_edge("accommodation_booking_agent", END)

# Set entry point to router
graph_v2.set_entry_point("router_agent")

# Compile the multi-agent graph
multi_agent_system = graph_v2.compile()

print("🏗️ Multi-agent system compiled!")
print("📊 Graph structure:")
print("   User → router_agent → [travel_info_agent | accommodation_booking_agent] → END")

## Step 17: Build Multi-Agent Graph

Now we'll create a more complex graph with routing capabilities.

In [ ]:
# Create the travel information agent
travel_info_agent_v2 = create_react_agent(
    model=llm_model_v2,
    tools=[search_travel_info, weather_forecast],  # Tools for travel info
    state_schema=AgentStateV2,
    prompt="You are a helpful travel assistant that can search travel information and get weather forecasts. "
           "Focus on destinations, attractions, activities, and weather in Cornwall, England. "
           "Only use the tools to find the information you need."
)

# Create the accommodation booking agent
accommodation_booking_agent = create_react_agent(
    model=llm_model_v2,
    tools=[check_bnb_availability, search_travel_info],  # Tools for booking
    state_schema=AgentStateV2,
    prompt="You are a helpful accommodation booking assistant. "
           "You can check BnB availability and prices in Cornwall. "
           "You can also search for general travel information to help with accommodation decisions. "
           "Always provide specific details about available rooms and pricing when possible."
)

print("✅ Specialized agents created!")
print("   🗺️ Travel Info Agent: search_travel_info + weather_forecast")
print("   🏨 Booking Agent: check_bnb_availability + search_travel_info")

## Step 16: Create Specialized Agents

Now we'll create two specialized agents using LangGraph's `create_react_agent` function.

In [ ]:
# Router system prompt
ROUTER_SYSTEM_PROMPT = (
    "You are a router. Given the following user message, decide if it is a travel information question "
    "(about destinations, attractions, or general travel info) "
    "or an accommodation booking question (about hotels, BnBs, room availability, or prices).\\n"
    "If it is a travel information question, respond with 'travel_info_agent'.\\n"
    "If it is an accommodation booking question, respond with 'accommodation_booking_agent'."
)

def router_agent_node(state: AgentStateV2) -> Command[AgentType]:
    \"\"\"
    Router node: decides which agent should handle the user query.
    
    This function:
    1. Extracts the user's message
    2. Uses the router LLM to classify the query type
    3. Returns a Command to route to the appropriate agent
    \"\"\"
    messages = state["messages"]
    last_msg = messages[-1] if messages else None
    
    if isinstance(last_msg, HumanMessage):
        user_input = last_msg.content
        print(f"🚦 Routing query: '{user_input[:50]}{'...' if len(user_input) > 50 else ''}'")
        
        # Create router messages
        router_messages = [
            SystemMessage(content=ROUTER_SYSTEM_PROMPT),
            HumanMessage(content=user_input)
        ]
        
        # Get routing decision from LLM
        router_response = llm_router.invoke(router_messages)
        agent_name = router_response.agent.value
        
        print(f"🎯 Routing to: {agent_name}")
        return Command(update=state, goto=agent_name)
    
    # Default to travel info agent if no clear user message
    return Command(update=state, goto=AgentType.travel_info_agent)

print("🚦 Router agent configured!")

## Step 15: Create Router Agent

The router agent analyzes user queries and decides which specialist agent should handle them.

In [ ]:
# Enhanced agent state for multi-agent system
class AgentStateV2(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    remaining_steps: RemainingSteps  # Track remaining steps in agent execution

# Define agent types
class AgentType(str, Enum):
    travel_info_agent = "travel_info_agent"
    accommodation_booking_agent = "accommodation_booking_agent"

# Structured output model for routing decisions
class AgentTypeOutput(BaseModel):
    agent: AgentType = Field(..., description="Which agent should handle the query?")

# Update tools list and create new LLM for multi-agent system
TOOLS_V2 = [search_travel_info, weather_forecast, check_bnb_availability]
llm_model_v2 = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create a structured LLM for routing decisions
llm_router = llm_model_v2.with_structured_output(AgentTypeOutput)

print("🎯 Agent routing system configured!")
print(f"📋 Available agents: {[agent.value for agent in AgentType]}")
print(f"🔧 Updated tools: {[tool.name for tool in TOOLS_V2]}")

## Step 14: Define Agent Types and Routing

We'll create an enum for agent types and a routing system to decide which agent should handle each query.

In [ ]:
# BnB booking data structures
class BnBOffer(TypedDict):
    bnb_id: int
    bnb_name: str
    town: str
    available_rooms: int
    price_per_room: float

class BnBBookingService:
    """Mock BnB booking service with sample accommodations across Cornwall."""
    
    @staticmethod
    def get_offers_near_town(town: str, num_rooms: int) -> List[BnBOffer]:
        """Get available BnB offers for a specific town and number of rooms."""
        # Mock BnB database with various accommodations
        mock_bnb_offers = [
            # Newquay
            {"bnb_id": 1, "bnb_name": "Seaside BnB", "town": "Newquay", "available_rooms": 3, "price_per_room": 80.0},
            {"bnb_id": 2, "bnb_name": "Surfside Guesthouse", "town": "Newquay", "available_rooms": 2, "price_per_room": 85.0},
            # Falmouth
            {"bnb_id": 3, "bnb_name": "Harbour View BnB", "town": "Falmouth", "available_rooms": 4, "price_per_room": 78.0},
            {"bnb_id": 4, "bnb_name": "Seafarer's Rest", "town": "Falmouth", "available_rooms": 1, "price_per_room": 90.0},
            # St Austell
            {"bnb_id": 5, "bnb_name": "Garden Gate BnB", "town": "St Austell", "available_rooms": 2, "price_per_room": 82.0},
            {"bnb_id": 6, "bnb_name": "Coastal Cottage BnB", "town": "St Austell", "available_rooms": 3, "price_per_room": 88.0},
            # Penzance
            {"bnb_id": 7, "bnb_name": "Penzance Pier BnB", "town": "Penzance", "available_rooms": 2, "price_per_room": 95.0},
            {"bnb_id": 8, "bnb_name": "Cornish Charm BnB", "town": "Penzance", "available_rooms": 3, "price_per_room": 87.0},
            # St Ives
            {"bnb_id": 9, "bnb_name": "St Ives Bay BnB", "town": "St Ives", "available_rooms": 3, "price_per_room": 97.0},
            {"bnb_id": 10, "bnb_name": "Artists' Retreat BnB", "town": "St Ives", "available_rooms": 2, "price_per_room": 102.0},
        ]
        
        # Filter offers by town and available rooms
        offers = [
            offer for offer in mock_bnb_offers 
            if offer["town"].lower() == town.lower() and offer["available_rooms"] >= num_rooms
        ]
        return offers

@tool(description="Check BnB room availability and price for a destination in Cornwall.")
def check_bnb_availability(destination: str, num_rooms: int) -> List[Dict]:
    """Check BnB room availability and price for the requested destination and number of rooms."""
    offers = BnBBookingService.get_offers_near_town(destination, num_rooms)
    if not offers:
        return [{"error": f"No available BnBs found in {destination} for {num_rooms} rooms."}]
    return offers

print("🏠 BnB booking tool added!")

## Step 13: Add BnB Booking Tool

Now let's add accommodation booking capabilities with a mock BnB service.

In [ ]:
# Weather forecast data structures
class WeatherForecast(TypedDict):
    town: str
    weather: Literal["sunny", "foggy", "rainy", "windy"]
    temperature: int

class WeatherForecastService:
    """Mock weather service that generates random but realistic weather data."""
    
    _weather_options = ["sunny", "foggy", "rainy", "windy"]
    _temp_min = 18  # Minimum temperature in Celsius
    _temp_max = 31  # Maximum temperature in Celsius

    @classmethod
    def get_forecast(cls, town: str) -> Optional[WeatherForecast]:
        """Generate a mock weather forecast for the given town."""
        weather = random.choice(cls._weather_options)
        temperature = random.randint(cls._temp_min, cls._temp_max)
        return WeatherForecast(town=town, weather=weather, temperature=temperature)

@tool(description="Get the weather forecast, given a town name.")
def weather_forecast(town: str) -> dict:
    """Get a mock weather forecast for a given town. Returns a WeatherForecast object with weather and temperature."""
    forecast = WeatherForecastService.get_forecast(town)
    if forecast is None:
        return {"error": f"No weather data available for '{town}'."}
    return forecast

print("🌤️ Weather forecasting tool added!")

## Step 12: Add Weather Forecasting Tool

Let's add a weather tool to demonstrate multiple tool capabilities.

In [ ]:
# Additional imports for multi-agent system
import random
from enum import Enum
from typing import Literal, Optional, List, Dict
from pydantic import BaseModel, Field

# LangGraph advanced features
from langgraph.managed.is_last_step import RemainingSteps
from langgraph.prebuilt import create_react_agent
from langgraph.types import Command
from langchain_core.messages import SystemMessage

print("✅ Enhanced dependencies loaded!")

# Part 2: Multi-Agent System with Routing 🚦

Now let's evolve our system to support multiple specialized agents with intelligent routing. We'll add:
- **Weather forecasting** capability
- **Hotel/BnB booking** functionality  
- **Router agent** that decides which specialist to use
- **Multiple specialized agents** for different tasks

## Step 11: Enhanced Dependencies

# LangGraph Multi-Agent System Tutorial 🤖

This notebook demonstrates the evolution of a **multi-agent travel assistant system** built with **LangGraph**. We'll progress from a simple single-agent setup to a sophisticated multi-agent system with routing, guardrails, and database integration.

## 🎯 What We'll Build
- **UK Travel Assistant** specializing in Cornwall destinations
- **Vector-based knowledge retrieval** from WikiVoyage pages  
- **Multi-agent architecture** with intelligent routing
- **Guardrails** for staying on-topic
- **Hotel booking integration** with SQLite database

## 📚 Learning Progression
1. **Part 1**: Basic single-agent with vector search (main_01_01.py)
2. **Part 2**: Multi-agent with routing (main_05_01.py)  
3. **Part 3**: Advanced system with guardrails & database (main_09_02.py)

---

# Part 1: Basic Single-Agent System 🚀

Let's start with the simplest version - a single agent that can search travel information using vector similarity search.

## Step 1: Import Dependencies

In [ ]:
# Core Python libraries
import os
import asyncio
import operator
import json
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv

# LangChain components for document processing and embeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# LangChain message types and tools
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

# LangGraph for building the agent workflow
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import tools_condition

print("✅ Dependencies imported successfully!")

## Step 2: Environment Setup

Load environment variables. You'll need an OpenAI API key to run this notebook.

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Check if OpenAI API key is available
if not os.environ.get("OPENAI_API_KEY"):
    print("⚠️ Warning: OPENAI_API_KEY not found in environment variables.")
    print("Please set your OpenAI API key before proceeding.")
    # Uncomment the line below to set it directly (not recommended for production)
    # os.environ["OPENAI_API_KEY"] = "your-key-here"
else:
    print("✅ OpenAI API key loaded successfully!")

## Step 3: Build Knowledge Base

We'll create a vector database from WikiVoyage pages about Cornwall destinations. This involves:
1. **Downloading** HTML pages from WikiVoyage
2. **Splitting** documents into chunks
3. **Embedding** chunks using OpenAI embeddings
4. **Storing** in Chroma vector database

In [ ]:
# Define the UK destinations we want to include in our knowledge base
UK_DESTINATIONS = [
    "Cornwall",
    "North_Cornwall", 
    "South_Cornwall",
    "West_Cornwall",
]

print(f"📍 Target destinations: {', '.join(UK_DESTINATIONS)}")

In [ ]:
async def build_vectorstore(destinations: Sequence[str]) -> Chroma:
    """
    Download WikiVoyage pages and create a Chroma vector store.
    
    This function:
    1. Constructs URLs for WikiVoyage pages
    2. Downloads HTML content asynchronously
    3. Splits documents into manageable chunks
    4. Creates embeddings and stores them in Chroma
    """
    # Step 1: Construct WikiVoyage URLs
    urls = [f"https://en.wikivoyage.org/wiki/{slug}" for slug in destinations]
    print(f"🌐 URLs to download: {urls}")
    
    # Step 2: Download pages asynchronously
    loader = AsyncHtmlLoader(urls)
    print("📥 Downloading destination pages...")
    docs = await loader.aload()
    print(f"📄 Downloaded {len(docs)} documents")
    
    # Step 3: Split documents into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1024,    # Each chunk will be ~1024 characters
        chunk_overlap=128   # 128 characters overlap between chunks
    )
    chunks = sum([splitter.split_documents([d]) for d in docs], [])
    print(f"✂️ Created {len(chunks)} text chunks")
    
    # Step 4: Create embeddings and store in vector database
    print(f"🧠 Embedding {len(chunks)} chunks...")
    vectordb_client = Chroma.from_documents(
        chunks, 
        embedding=OpenAIEmbeddings()
    )
    print("✅ Vector store ready!")
    
    return vectordb_client

In [ ]:
# Build the vector store (this will take a moment)
print("🏗️ Building vector store...")
ti_vectorstore_client = await build_vectorstore(UK_DESTINATIONS)
ti_retriever = ti_vectorstore_client.as_retriever()
print("\n🎉 Knowledge base ready for queries!")